# UNet + Transformer baseline — inference & submission

Runs the trained baseline on the test volumes and writes `submission.csv`.

It runs with **no internet**: the model weights, repo source, and dependency wheels all come from the artifacts dataset.

> **Setup before running:**
> 1. Enable the **GPU** accelerator; leave **Internet OFF**.
> 2. **Add Input** → the competition data *and* your `cellmot-baseline-artifacts` dataset (from the train notebook's output).
> 3. Set `ARTIFACTS` below to match the dataset's mount path.

## 1. Configuration

In [ ]:
import os, glob

# --- Robust, mount-agnostic path discovery (pushed-kernel mounts differ from the
# interactive editor: the attached dataset mounts at /kaggle/input/<slug>/... and
# competition data at /kaggle/input/<comp>/... — so we DISCOVER, never hardcode). ---
def _find_artifacts():
    for p in sorted(glob.glob("/kaggle/input/**/cellmot-baseline-artifacts", recursive=True), key=len):
        if os.path.isdir(os.path.join(p, "wheels")) and os.path.isdir(os.path.join(p, "repo")):
            return p
    raise FileNotFoundError("cellmot-baseline-artifacts (wheels+repo) not found under /kaggle/input")

def _find_test_dir():
    # Prefer a dir literally named 'test' holding *.zarr (the competition test split).
    named = [p for p in glob.glob("/kaggle/input/**/test", recursive=True)
             if os.path.isdir(p) and glob.glob(os.path.join(p, "*.zarr"))
             and "cellmot-baseline-artifacts" not in p]
    if named:
        return sorted(named, key=len)[0]
    # Fallback: any dir containing *.zarr videos that is not the artifacts dataset.
    for z in sorted(glob.glob("/kaggle/input/**/*.zarr", recursive=True), key=len):
        d = os.path.dirname(z)
        if "cellmot-baseline-artifacts" not in d:
            return d
    raise FileNotFoundError("no competition test *.zarr videos found under /kaggle/input")

ARTIFACTS = _find_artifacts()
TEST_DIR = _find_test_dir()
print("ARTIFACTS =", ARTIFACTS)
print("TEST_DIR  =", TEST_DIR)

REPO_DIR = "/kaggle/working/repo"
METHOD = "unet_transformer"

# Model checkpoint (relative to the repo). Released split_0 weights.
WEIGHTS = f"weights/{METHOD}/split_0/edge_predictor_best.pth"

# Detection peak threshold. GT is sparse; ~0.99 scored best in the sweep
# (SOT-3020 probe freezes the documented candidate config verbatim).
DET_THRESHOLD = 0.99

UNET_BATCH_SIZE = 4          # frame-pairs per UNet forward; lower if OOM
SLICE = ""                   # "" = all test videos

# SOT-3011 WHOLESALE learned pipeline: learned detection + learned ILP linking
# (global, flow-consistent). This is the organiser's tested offline baseline.
USE_ILP = True
ILP_EDGE_WEIGHT = -1.0
ILP_APPEARANCE_WEIGHT = 0.1
ILP_DISAPPEARANCE_WEIGHT = 0.1
ILP_DIVISION_WEIGHT = 1.0


## 2. Offline install & setup

Install the dependencies from the bundled wheels (no internet), then copy the repo source and weights into a writable location and put the package on the import path.

In [ ]:
import sys
import shutil
import subprocess

subprocess.run(
    ["pip", "install", "--no-index", "--find-links", f"{ARTIFACTS}/wheels",
     "tracksdata", "zarr>=3.0.10", "pyscipopt"],
    check=True,
)

shutil.copytree(f"{ARTIFACTS}/repo", REPO_DIR, dirs_exist_ok=True)
shutil.copytree(f"{ARTIFACTS}/weights", f"{REPO_DIR}/weights", dirs_exist_ok=True)
sys.path.insert(0, f"{REPO_DIR}/src")

print("Weights:", os.listdir(f"{REPO_DIR}/weights/{METHOD}/split_0"))

## 3. Inference on all test videos

Build a one-fold splits file listing every test video, then run prediction. `PYTHONPATH=src` makes the `tracking_cellmot` package importable without an internet install. Each video's tracks are exported as a `.geff` graph.

In [ ]:
import json

test_stems = sorted(f[:-5] for f in os.listdir(TEST_DIR) if f.endswith(".zarr"))
print(f"{len(test_stems)} test videos")

with open(f"{REPO_DIR}/kaggle_test_splits.json", "w") as f:
    json.dump([{"split": 0, "train": [], "test": test_stems}], f)

In [ ]:
import subprocess

cmd = [
    "python", "scripts/predict_unet_transformer.py",
    "--data-dir", TEST_DIR, "--splits", "kaggle_test_splits.json", "--split", "0",
    "--weights", WEIGHTS, "--unet-batch-size", str(UNET_BATCH_SIZE),
    "--det-threshold", str(DET_THRESHOLD),
    "--ilp-edge-weight", str(ILP_EDGE_WEIGHT),
    "--ilp-appearance-weight", str(ILP_APPEARANCE_WEIGHT),
    "--ilp-disappearance-weight", str(ILP_DISAPPEARANCE_WEIGHT),
    "--ilp-division-weight", str(ILP_DIVISION_WEIGHT),
]
if USE_ILP:
    cmd.append("--use-ilp")
if SLICE:
    cmd += ["--slice", SLICE]

print(" ".join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, env={**os.environ, "PYTHONPATH": "src"}, check=True)

## 4. Build `submission.csv`

Flatten the predicted `.geff` graphs into the competition's CSV format: one `node` row per detection (`t, z, y, x`) and one `edge` row per link (`source_id, target_id`).



In [ ]:
from pathlib import Path

import pandas as pd
import tracksdata as td

geffs = sorted(Path(REPO_DIR, "predictions").glob(f"*/{METHOD}/split_0/*.geff"))
print(f"{len(geffs)} prediction graphs")

rows = []
for g in geffs:
    name = g.stem
    graph = td.graph.IndexedRXGraph.from_geff(g)
    graph = graph[0] if isinstance(graph, tuple) else graph
    for r in graph.node_attrs().iter_rows(named=True):
        rows.append({
            "dataset": name, "row_type": "node", "node_id": int(r["node_id"]),
            "t": int(r["t"]), "z": int(round(r["z"])), "y": int(round(r["y"])),
            "x": int(round(r["x"])), "source_id": -1, "target_id": -1,
        })
    for r in graph.edge_attrs().iter_rows(named=True):
        rows.append({
            "dataset": name, "row_type": "edge", "node_id": -1,
            "t": -1, "z": -1, "y": -1, "x": -1,
            "source_id": int(r["source_id"]), "target_id": int(r["target_id"]),
        })

submission = pd.DataFrame(
    rows,
    columns=["dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"],
)
submission.index.name = "id"
submission.to_csv("submission.csv")
print(f"Wrote submission.csv with {len(submission)} rows")
submission.head()

## Submitting

**Save Version → Save & Run All** (Internet off), then **Submit** the generated `submission.csv` to the competition to appear on the leaderboard.

To improve on this baseline: train longer (raise `EPOCHS` in the train notebook), tune detection/linking, or improve division handling — then regenerate the artifacts dataset and re-run this notebook. See the [repository](https://github.com/royerlab/kaggle-cell-tracking-competition) for the model and full API.